### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [ ]:
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("What is agentic AI?")
response

AIMessage(content='**Agentic AI** is a class of artificial intelligence systems that are designed to act autonomously, pursue goals, and make decisions in a way that is *agent-like*—much like a human or animal agent would. The term “agentic” comes from the word *agency*, meaning the capacity to act independently and make choices.\n\n---\n\n## Core Characteristics\n\n| Feature | What It Means | Typical Techniques |\n|---------|---------------|---------------------|\n| **Goal‑oriented behavior** | The system has explicit objectives to achieve (e.g., “maximize profit,” “avoid collisions”). | Reward functions in reinforcement learning (RL), utility functions in decision theory. |\n| **Autonomy** | It can operate without continuous human supervision, making decisions in real time. | Model‑free RL, hierarchical planning, meta‑learning. |\n| **Adaptive planning** | It can devise and revise plans to reach goals, given changing environments. | Graph‑based planners, Monte Carlo Tree Search, Baye

In [ ]:
from langchain.tools import tool


@tool
def get_exchange_rate(currency_pair:str)->str:
    """Get the exchange rate for a currency pair, e.g. 'USD_EUR'"""
    rates = {
        "USD_EUR": "0.92",
        "USD_GBP": "0.78",
        "EUR_GBP": "0.85",
    }
    return rates.get(currency_pair, f"No exchange rate available for {currency_pair}")


model_with_tools=model.bind_tools([get_exchange_rate]) #bind_tools only tells the model that the tool exists (its name, description, and argument schema).

In [3]:
response = model_with_tools.invoke("What's the exchange rate for USD to EUR?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks: "What\'s the exchange rate for USD to EUR?" We need to use the get_exchange_rate function. The pair is "USD_EUR". We\'ll call the function.', 'tool_calls': [{'id': 'fc_5c079a35-3463-444c-a109-adc87cc990d6', 'function': {'arguments': '{"currency_pair":"USD_EUR"}', 'name': 'get_exchange_rate'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 141, 'total_tokens': 206, 'completion_time': 0.068939961, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_time': 0.006972773, 'prompt_tokens_details': None, 'queue_time': 0.284884415, 'total_time': 0.075912734}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a202fe-0900-747b-9373-49dc0cbf08c2-0' tool_calls=[{'name': 'get_exchange_rate', 'args': {'currency_pair': 'USD_EUR'}, 'i

### Tool Execution Loops

In [4]:
messages = [{"role": "user", "content": "What's the exchange rate for USD to EUR?"}]
ai_msg = model_with_tools.invoke(messages)

In [6]:
ai_msg.tool_calls

[{'name': 'get_exchange_rate',
  'args': {'currency_pair': 'USD_EUR'},
  'id': 'fc_2fcdab7a-8591-412d-9646-8621a12600ef',
  'type': 'tool_call'}]

In [8]:
count=0
for i in ai_msg.tool_calls:
    count=count+1
    print(i,count)

{'name': 'get_exchange_rate', 'args': {'currency_pair': 'USD_EUR'}, 'id': 'fc_2fcdab7a-8591-412d-9646-8621a12600ef', 'type': 'tool_call'} 1


In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the exchange rate for USD to EUR?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_exchange_rate.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The exchange rate for USD to EUR is 0.92."

In [ ]:
messages

### 🤔 Before you move on
Print the `messages` list from the cell above — it's not just plain strings, it's a mix of typed objects (a human turn, an AI turn carrying tool_calls, a tool result). What exactly are these message types, and how is a `SystemMessage` different from a `HumanMessage`, `AIMessage`, or `ToolMessage`? That's the whole subject of `4-messages.ipynb`.